# 01 · Análisis exploratorio de datos (EDA)
**Pregunta:** ¿qué distingue a los estudiantes con reserva aprobada que no concretan su matrícula entre el 20 de febrero y el 30 de abril?

Con `FUENTE = "real"` (solo en entorno autorizado) todas las tablas y gráficos ocultan los grupos con menos de 5 casos. Las figuras se guardan en `results/figures/` a 300 DPI, con el prefijo de la fuente.

Incluye las dos auditorías **obligatorias de Sprint 1** (Checklist SMART y Ficha técnica): el conteo de eventos de no matrícula por estudiante y por representante, y la comparabilidad de C1 (secciones 3 y 9).

In [ ]:
# @title 0 · Preparar el entorno (ejecutar primero)
# Si el repositorio ya está en GitHub, pegue su URL; si no, se pedirá subir el .zip.
REPO_URL = ""  # @param {type:"string"}

import importlib.util, os, subprocess, sys, zipfile
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    RAIZ = Path("/content/prediccion-matricula-lemas")
    if not RAIZ.exists():
        if REPO_URL:
            subprocess.run(["git", "clone", "-q", REPO_URL, str(RAIZ)], check=True)
        else:
            from google.colab import files
            print("Suba el archivo prediccion-matricula-lemas.zip")
            nombre = next(iter(files.upload()))
            zipfile.ZipFile(nombre).extractall("/content")
else:
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
REQUERIDOS = {'pandas': 'pandas', 'matplotlib': 'matplotlib', 'seaborn': 'seaborn', 'yaml': 'PyYAML', 'sklearn': 'scikit-learn'}  # módulo -> paquete pip
faltan = [pip for modulo, pip in REQUERIDOS.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)
print("Entorno:", "Google Colab" if EN_COLAB else "local", "| Raíz:", RAIZ)

In [ ]:
# @title Cargar la base
FUENTE = "sintetica"  # @param ["sintetica", "real"]
# Marque SUBIR_DE_NUEVO si subió un archivo equivocado (se borra y se vuelve a pedir)
SUBIR_DE_NUEVO = False  # @param {type:"boolean"}

import pandas as pd
from src.data_processing import construir_dataset, limpiar_base
from src.utils import cargar_config, configurar_logging, obtener_base_seud

configurar_logging()
config = cargar_config()
MIN = config["privacidad"]["min_celda"]
ruta = RAIZ / config["rutas"]["base_real" if FUENTE == "real" else "base_sintetica"]

crudo = (obtener_base_seud(ruta, EN_COLAB, SUBIR_DE_NUEVO) if FUENTE == "real"
         else pd.read_csv(ruta, dtype=str))
base = limpiar_base(crudo, config)
dataset, reporte = construir_dataset(base, config)
if dataset.empty or "y_no_matricula" not in dataset:
    anios = crudo["anoa"].astype(str).str.strip().value_counts().head(10).to_dict()
    raise RuntimeError(
        "No se pudo construir ninguna cohorte. Revise la columna anoa: debe tener el año "
        f"de inicio del ciclo (2021…2026). Valores más frecuentes: {anios}")
print(f"Fuente: {FUENTE} | filas base: {len(base)} | filas modelables: {len(dataset)}")
if FUENTE == "sintetica":
    print("⚠️ Datos SINTÉTICOS: sirven para probar el código, no describen a LEMAS.")

In [ ]:
# @title Utilidades de gráficos
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils import tasa_por_grupo

sns.set_theme(style="whitegrid", palette="deep")
FIG = RAIZ / config["rutas"]["figuras"]
FIG.mkdir(parents=True, exist_ok=True)
TASA_GLOBAL = dataset["y_no_matricula"].mean()

def guardar(fig, nombre):
    fig.tight_layout()
    fig.savefig(FIG / f"{FUENTE}_{nombre}.png", dpi=config["figuras"]["dpi"], bbox_inches="tight")
    plt.show()

def barras_tasa(ax, col, titulo, orden=None):
    t = tasa_por_grupo(dataset, col, minimo=MIN)
    if orden is not None:
        t = t.set_index(col).reindex([o for o in orden if o in set(t[col])]).reset_index()
    ax.bar(t[col].astype(str), 100 * t["tasa"], color="#8B1E3F")
    ax.axhline(100 * TASA_GLOBAL, ls="--", c="gray", label="tasa global")
    for i, (tasa, n) in enumerate(zip(t["tasa"], t["n"])):
        ax.text(i, 100 * tasa, f"n={n}", ha="center", va="bottom", fontsize=8)
    ax.set(title=titulo, ylabel="% no matrícula", xlabel="")
    ax.tick_params(axis="x", rotation=30)
    for etiqueta in ax.get_xticklabels():
        etiqueta.set_horizontalalignment("right")
    return t

## 1. Calidad de los datos

In [ ]:
calidad = pd.DataFrame({"% nulos": (100 * base.isna().mean()).round(2), "tipo": base.dtypes.astype(str)})
display(calidad[calidad["% nulos"] > 0].sort_values("% nulos", ascending=False))
print("Filas por año:", base["anio_origen"].value_counts().sort_index().to_dict())

## 2. Embudo de elegibilidad por cohorte

In [ ]:
columnas = ["cohorte", "origen_total", "sin_reserva", "pendientes", "aprobados",
            "reserva_despues_t0", "terminales_excluidos", "confirmados_antes_t0", "modelables",
            "no_matricula", "tardias", "matriculados_sin_reserva_aprobada"]
embudo = reporte[columnas].set_index("cohorte")
if FUENTE == "real":
    embudo = embudo.apply(lambda s: s.astype(object).where(s >= MIN, f"<{MIN}"))
display(embudo)

fig, ax = plt.subplots(figsize=(9, 4))
r = reporte.set_index("cohorte")
ax.bar(r.index, r["modelables"], label="modelables", color="#C9D6E3")
ax.bar(r.index, r["no_matricula"], label="no matrícula", color="#8B1E3F")
ax.set(title="Figura 1. Población modelable y casos de no matrícula por cohorte", ylabel="estudiantes")
ax.legend()
guardar(fig, "01_embudo_cohortes")

## 3. Conteo de eventos de no matrícula (auditoría obligatoria)
Muestra, para cada cohorte, cuántos casos de no matrícula hay por estudiante y por representante. Una cohorte se considera **suficiente** si alcanza el mínimo configurado en `config.yaml` (`auditoria.min_eventos_por_cohorte`). La regla EPV (eventos por predictor) indica si el entrenamiento tiene suficientes casos para un modelo lineal estable.

In [ ]:
from src.auditoria import conteo_eventos, eventos_por_predictor

conteo = conteo_eventos(dataset, config)
vista = conteo.copy()
if FUENTE == "real":
    for col in ["estudiantes", "eventos_estudiante", "familias", "eventos_familia"]:
        vista[col] = vista[col].astype(object).where(vista[col] >= MIN, f"<{MIN}")
display(vista)

epv_minimo = config["auditoria"]["eventos_por_predictor"]
for nombre, cohortes in config["auditoria"]["configuraciones_entrenamiento"].items():
    epv = eventos_por_predictor(dataset, cohortes)
    estado = "✅" if epv >= epv_minimo else "⚠️ bajo: preferir modelos regularizados y simples"
    print(f"EPV {nombre} ({'+'.join(cohortes)}): {epv} {estado}")

## 4. Balance de clases y estabilidad entre cohortes

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
t = barras_tasa(ax, "cohorte", "Figura 2. Tasa de no matrícula por cohorte")
guardar(fig, "02_tasa_cohorte")
print(f"Tasa global: {100 * TASA_GLOBAL:.1f}% → clases desbalanceadas; "
      "se usará class_weight='balanced' y métricas centradas en la clase positiva.")

## 5. Riesgo según variables categóricas

In [ ]:
dataset["conducta_letra"] = dataset["conducta"].map({5: "A", 4: "B", 3: "C", 2: "D", 1: "E"})
fig, ejes = plt.subplots(2, 3, figsize=(15, 8))
barras_tasa(ejes[0, 0], "sede", "Sede")
barras_tasa(ejes[0, 1], "subnivel", "Subnivel")
barras_tasa(ejes[0, 2], "pago_origen", "Pago de la matrícula anterior")
barras_tasa(ejes[1, 0], "beca", "Beca (1 = sí)")
barras_tasa(ejes[1, 1], "reserva_extraordinaria", "Reserva extraordinaria (1 = sí)")
barras_tasa(ejes[1, 2], "conducta_letra", "Conducta", orden=list("ABCDE"))
fig.suptitle("Figura 3. Tasa de no matrícula por variable categórica", y=1.02)
guardar(fig, "03_tasa_categoricas")

## 6. Pensiones pagadas tarde y hermanos en LEMAS

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13, 4))
barras_tasa(ejes[0], "atrasos_pension", "Pensiones pagadas tarde en el ciclo", orden=list(range(11)))
barras_tasa(ejes[1], "hermanos_lemas", "Hermanos matriculados en LEMAS")
fig.suptitle("Figura 4. Riesgo según atrasos en pensiones y hermanos", y=1.02)
guardar(fig, "04_atrasos_hermanos")

## 7. Variables numéricas según resultado

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
etiquetas = dataset["y_no_matricula"].map({0: "se matriculó", 1: "no matrícula"})
sns.boxplot(x=etiquetas, y=dataset["promedio"], ax=ejes[0], color="#C9D6E3")
ejes[0].set(title="Promedio académico", xlabel="", ylabel="promedio (0-10)")
sns.boxplot(x=etiquetas, y=dataset["anios_permanencia"], ax=ejes[1], color="#C9D6E3")
ejes[1].set(title="Años de permanencia en LEMAS", xlabel="", ylabel="años")
fig.suptitle("Figura 5. Distribución de variables numéricas por resultado", y=1.02)
guardar(fig, "05_numericas_resultado")

## 8. Correlaciones (Spearman)

In [ ]:
numericas = ["promedio", "conducta", "anios_permanencia", "atrasos_pension", "hermanos_lemas",
             "beca", "reserva_extraordinaria", "y_no_matricula"]
corr = dataset[numericas].astype(float).corr(method="spearman")
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Figura 6. Correlación de Spearman entre variables y el resultado")
guardar(fig, "06_correlaciones")

## 9. Deriva entre años (cambio de escala de conducta, pospandemia)

In [ ]:
deriva = dataset.groupby("cohorte").agg(promedio=("promedio", "mean"),
                                         atrasos=("atrasos_pension", "mean"),
                                         conducta=("conducta", "mean"), beca=("beca", "mean"),
                                         n=("y_no_matricula", "size")).round(3)
display(deriva[deriva["n"] >= MIN])
fig, ax = plt.subplots(figsize=(9, 4))
pd.crosstab(dataset["cohorte"], dataset["conducta_letra"], normalize="index")[
    [c for c in "ABCDE" if c in set(dataset["conducta_letra"].dropna())]].plot.bar(stacked=True, ax=ax, colormap="RdYlGn_r", rot=0)
ax.set(title="Figura 7. Distribución de conducta por cohorte", ylabel="proporción", xlabel="")
ax.legend(title="Conducta", bbox_to_anchor=(1, 1))
guardar(fig, "07_deriva_conducta")

## 10. Comparabilidad de C1 frente a C2–C4 (diferencias estandarizadas)
C1 (2021–2022) puede diferir por el contexto de pospandemia y por la conversión de la conducta a letras. Se compara cada predictor con C2–C4 (v5, sección 5.3.4) mediante la **diferencia de medias estandarizada (SMD)**; un |SMD| > 0,20 señala un cambio relevante. La decisión final de incluir C1 se toma en el notebook 02, evaluando en C4.

In [ ]:
from src.auditoria import diferencias_estandarizadas

smd = diferencias_estandarizadas(dataset, config)
display(smd.head(15))
print(f"Variables con |SMD| > {config['auditoria']['umbral_smd']}: {int(smd['relevante'].sum())}")

top = smd.head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 5))
colores = ["#8B1E3F" if r else "#9DB4C9" for r in top["relevante"]]
etiquetas = [f"{v} = {n}" if n else v for v, n in zip(top["variable"], top["nivel"])]
ax.barh(etiquetas, top["smd"], color=colores)
umbral = config["auditoria"]["umbral_smd"]
for x in (-umbral, umbral):
    ax.axvline(x, ls="--", c="gray")
ax.set(title="Figura 8. Diferencias estandarizadas de C1 frente a C2–C4", xlabel="SMD")
guardar(fig, "08_comparabilidad_c1")

## 11. Hallazgos
*Completar tras ejecutar con `FUENTE = "real"`. Estas conclusiones se trasladan a `docs/analisis_datos.md`.*

1. **Balance de clases:** …
2. **Variables con mayor diferencia de riesgo:** …
3. **Estabilidad entre cohortes:** …
4. **Suficiencia de eventos por cohorte y EPV:** …
5. **Comparabilidad de C1:** …
6. **Calidad de datos y decisiones de limpieza:** …

In [ ]:
# @title Descargar las figuras (Colab)
import shutil
paquete = shutil.make_archive(str(RAIZ / f"figuras_{FUENTE}"), "zip", FIG)
print("Figuras:", sorted(p.name for p in FIG.glob(f"{FUENTE}_*.png")))
if EN_COLAB:
    from google.colab import files
    files.download(paquete)